# AutoDL v6 overlap40：DeepLabV3+-ResNet50 seed1337

A1337 的同种子、同环境结构对照。physical batch4、80 epochs、ImageNet初始化、Val前景mIoU选模，不访问 Test。

## 1. 服务器路径

In [ ]:
from pathlib import Path

DATA_ROOT = Path('/root/autodl-tmp/datasets/dataset_v6_random811_overlap40')
OUTPUT_ROOT = Path('/root/autodl-tmp/outputs')
REPO_DIR = Path('/root/autodl-tmp/projects/lunar-linear')
HF_CACHE_SOURCE = Path('/root/autodl-tmp/resnet50_imagenet_cache')
CONFIG_FILE = 'v6_overlap40_deeplab_batch4_seed1337.json'

## 2. 环境、离线 ImageNet 权重和输入核验

In [ ]:
import hashlib, importlib.metadata, importlib.util, json, os, shutil, subprocess, sys

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
required = [('rasterio', 'rasterio'), ('tqdm', 'tqdm')]
missing = [package for module, package in required if importlib.util.find_spec(module) is None]
try:
    smp_version = importlib.metadata.version('segmentation-models-pytorch')
except importlib.metadata.PackageNotFoundError:
    smp_version = None
if smp_version != '0.5.0':
    missing.append('segmentation-models-pytorch==0.5.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
import numpy as np
import torch
assert torch.cuda.is_available(), '服务器未识别 GPU'
assert (REPO_DIR / '.git').is_dir(), f'仓库不存在: {REPO_DIR}'
PROJECT_DIR = REPO_DIR / 'LTL-Net'
config = json.loads((PROJECT_DIR / 'configs' / CONFIG_FILE).read_text(encoding='utf-8'))
assert config['module'] == 'deeplab' and config['seed'] == 1337
assert config['epochs'] == 80 and config['batch_size'] == 4 and config['accum_steps'] == 1
assert config['learning_rate'] == 5e-5 and config['encoder_weights'] == 'imagenet'
assert config['boundary_weight'] == 0.0 and config['automatic_test_evaluation'] is False
expected_cache_hashes = {
    'config.json': '01bf2cf24eb29b405c28c159f46ceda92c098ab85868be88fb100967db47166e',
    'model.safetensors': 'df1aad85e18536504a4c8597118364e291ff3a9c4b56dd9b3a4900642e4c3a7c',
}
snapshot = Path('/root/.cache/huggingface/hub/models--smp-hub--resnet50.imagenet/snapshots/00cb74e366966d59cd9a35af57e618af9f88efe9')
snapshot.mkdir(parents=True, exist_ok=True)
for name, expected_hash in expected_cache_hashes.items():
    source = HF_CACHE_SOURCE / name
    target = snapshot / name
    assert source.is_file(), f'离线ImageNet权重文件不存在: {source}'
    source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
    assert source_hash == expected_hash, (name, source_hash, expected_hash)
    if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected_hash:
        shutil.copy2(source, target)
os.environ['HF_HUB_OFFLINE'] = '1'
assert DATA_ROOT.is_dir(), f'数据目录不存在: {DATA_ROOT}'
for split, expected in config['expected_tiles'].items():
    images = sorted([*(DATA_ROOT / split / 'image').glob('*.tif'), *(DATA_ROOT / split / 'image').glob('*.tiff')])
    masks = sorted([*(DATA_ROOT / split / 'mask').glob('*.tif'), *(DATA_ROOT / split / 'mask').glob('*.tiff')])
    assert len(images) == len(masks) == expected, (split, len(images), len(masks))
    assert {p.stem for p in images} == {p.stem for p in masks}
for name, expected_hash in config['expected_metadata_sha256'].items():
    actual_hash = hashlib.sha256((DATA_ROOT / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, (name, actual_hash, expected_hash)
stats = json.loads((DATA_ROOT / 'normalization_stats.json').read_text(encoding='utf-8'))
assert np.allclose(stats['mean'], config['expected_mean'], rtol=0, atol=1e-12)
assert np.allclose(stats['std'], config['expected_std'], rtol=0, atol=1e-12)
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('GPU:', torch.cuda.get_device_name(0))
print('Commit:', commit)
print('Data:', DATA_ROOT)
print('Offline ImageNet snapshot:', snapshot)
print(json.dumps(config, ensure_ascii=False, indent=2))

## 3. ImageNet初始化与 batch4 冒烟

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
from models.module_models import build_module_model
from train_module_experiment import ExperimentLoss, model_outputs
smoke_model = build_module_model('deeplab', encoder_weights='imagenet').cuda().train()
criterion = ExperimentLoss('deeplab', boundary_weight=0.0).cuda()
optimizer = torch.optim.AdamW(smoke_model.parameters(), lr=config['learning_rate'])
x = torch.randn(4, 5, 512, 512, device='cuda')
labels = torch.randint(0, 5, (4, 512, 512), device='cuda')
with torch.amp.autocast('cuda'):
    logits, boundary_logits = model_outputs(smoke_model, x, False)
    loss, _ = criterion(logits, labels, boundary_logits)
loss.backward(); optimizer.step()
assert logits.shape == (4, 5, 512, 512) and torch.isfinite(loss)
print('Smoke loss:', float(loss))
del smoke_model, criterion, optimizer, x, labels, logits, loss
torch.cuda.empty_cache()

## 4. 正式训练 DeepLab1337

In [ ]:
result_dir = OUTPUT_ROOT / f"result_{config['run_name']}"
assert not result_dir.exists(), f'结果目录已存在，为防止覆盖已停止: {result_dir}'
command = [
    sys.executable, str(PROJECT_DIR / 'scripts' / 'train_module_experiment.py'),
    '--module', config['module'], '--data-dir', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT), '--run-name', config['run_name'],
    '--seed', str(config['seed']), '--epochs', str(config['epochs']),
    '--batch-size', str(config['batch_size']), '--accum-steps', str(config['accum_steps']),
    '--num-workers', str(config['num_workers']), '--learning-rate', str(config['learning_rate']),
    '--boundary-weight', str(config['boundary_weight']), '--encoder-weights', config['encoder_weights'],
]
print(' '.join(command))
subprocess.check_call(command, cwd=PROJECT_DIR)

## 5. 查看并打包结果

In [ ]:
result = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
assert result['test_evaluated'] is False
assert result['seed'] == 1337 and result['model'] == 'deeplab'
print(json.dumps(result, ensure_ascii=False, indent=2))
archive_path = shutil.make_archive(str(OUTPUT_ROOT / result_dir.name), 'zip', root_dir=result_dir)
print('结果目录:', result_dir)
print('压缩包:', archive_path)